# mcp

> Ramabana's tools, and Ramabana's whole agent, as an MCP server.

Two different things to hand another agent, and the difference matters. Mounting the *tools* gives it hash-verified editing, a code index and notebook-aware cell addressing over your folders. Mounting the *agent* gives it one tool, `ask`, that runs a whole Ramabana turn and returns only the answer. This is the same context argument as `delegate_search`, one layer out.

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import expect_fail, test_eq
from ramabana.testing import FullHost, fake_agent

In [ ]:
#| export
import json
from pathlib import Path
import anyio
from fastcore.script import call_parse
from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations
from ramabana.tools import WRITE_TOOLS, LocalHost, discover, find, skill_index, tools_for
from ramabana.cli import mk_agent, mk_host
from ramabana.core import PII_OFF

## The server

With an agent, the tools mounted here are the very objects a turn gets. Recorded tool wrappers share the agent activity log. A client's call lands in the same activity feed and the same `changes()` report as a turn. `functools.wraps` in `Agent._record` kept their signatures and docstrings, and that is exactly what FastMCP reads to build a schema. There is no second description of any tool anywhere and nothing to drift. Without an agent the tools are built straight off the host.

`readonly` defaults to True because the client is another agent whose approval UI this server does not control. Writes are one flag away, and should be gated the usual way when they are mounted.

In [ ]:
#| export
UNSAFE = WRITE_TOOLS | {'delegate_search', 'delegate_parallel'}

INSTRUCTIONS = """Ramabana's tools over the folders this server was started on.

`search_code` and `view_file` are the way in: file views are `lineno|hash|content`, and those
hashes are the addresses `edit_file` takes, so an edit built on a stale view fails instead of
damaging the wrong line. Notebooks are addressed by cell id (`notebook_cells`, then
`view_cell`/`edit_cell`), never by line. `ask` hands a whole task to Ramabana's own agent and
returns just its answer."""


def _annotate(name):
    "MCP's hints about a tool. A client can style and gate it without guessing from the name."
    return ToolAnnotations(readOnlyHint=name not in UNSAFE,
                           destructiveHint=name in WRITE_TOOLS,
                           idempotentHint=name not in WRITE_TOOLS,
                           openWorldHint=name in ('web_search', 'read_url', 'research'))

In [ ]:
#| export
def server(host=None, agent=None, name='ramabana', readonly=True, delegate=True, **kw):
    """An MCP server over one host's tools, and optionally over Ramabana's own agent.

    The tools are the same objects the model gets in a turn. FastMCP reads their own
    signatures. `readonly=True` omits write tools by default because the client cannot use this process's approval UI.
    """
    host = host if host is not None else LocalHost()
    mcp = FastMCP(name, instructions=INSTRUCTIONS, **kw)
    skills = discover(host.roots, getattr(agent, 'cfg', None)) if agent is None else agent.skills
    # the agent's own recorded tools when there is one. A client's call reaches its feed
    every = agent.tools if agent is not None else tools_for(host, get_skills=lambda: skills)
    mounted = []
    for tool in every:
        nm = getattr(tool, '__name__', '')
        if readonly and nm in UNSAFE: continue
        mcp.add_tool(tool, annotations=_annotate(nm))
        mounted.append(nm)

    @mcp.resource('skill://{name}', mime_type='text/markdown')
    def skill(name: str) -> str:
        "One skill in full, by name. The same text `read_skill` returns in a turn."
        s = find(skills, name)
        return f'no skill matching {name!r}' if s is None else s.text()

    @mcp.resource('ramabana://skills', mime_type='text/markdown')
    def skill_list() -> str:
        "Every skill this server can read, with the one-line description of when it applies."
        return skill_index(skills) or 'no skills found'

    @mcp.resource('ramabana://status', mime_type='application/json')
    def status() -> str:
        "What is loaded: folders, model, tools, skills, and anything that has gone wrong."
        out = {'roots': list(host.roots), 'tools': mounted, 'skills': [s.name for s in skills]}
        if agent is not None: out['agent'] = agent.status()
        return json.dumps(out, default=str, indent=2)

    if agent is not None and delegate: _mount_ask(mcp, agent)
    return mcp

In [ ]:
#| export
def _mount_ask(mcp, agent):
    "Add the `ask` tool: one whole Ramabana turn, with only its answer coming back."

    @mcp.tool(annotations=ToolAnnotations(readOnlyHint=False, openWorldHint=True))
    async def ask(task: str) -> str:
        """Hand a whole task to Ramabana's agent and get back only its answer.

        Use this when answering would take many tool calls whose results you do not need to
        keep. "where else do we do X", "what does this module actually do", "make this
        change and tell me what you changed". Ramabana runs its own tool loop with its own
        model. This costs you one question and one answer.

        Ask one self-contained task: this agent cannot see your conversation.
        """
        return await anyio.to_thread.run_sync(agent.ask, task)

A server over a host with every capability. Read-only by default. `edit_file` and `create_file` are simply not there:

In [ ]:
host = FullHost(files={'pkg/sizes.py': 'RESERVE = 16_384\n\ndef threshold(ctx):\n    return ctx - RESERVE\n'})
mcp = server(host)
tools = await mcp.list_tools()
[t.name for t in tools]

['search_code',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'notebook_cells',
 'view_cell',
 'web_search',
 'read_url',
 'research',
 'memory_search',
 'memory_tree',
 'memory_read',
 'memory_topics',
 'list_vars',
 'scale_numeric',
 'inspect_python',
 'read_terminal',
 'read_skill']

In [ ]:
test_eq(set(t.name for t in tools) & WRITE_TOOLS, set())
test_eq('view_file' in [t.name for t in tools], True)
len(tools)

19

Each tool arrives with the schema FastMCP derived from the function itself. The description is the docstring the model reads in a turn, and the parameters are the annotations:

In [ ]:
t = next(t for t in tools if t.name == 'view_file')
t.description.splitlines()[0], t.inputSchema['properties'], t.annotations.readOnlyHint

('Read a file as `lineno|hash|content` lines. Optionally limit to lines `start`..`end`.',
 {'path': {'title': 'Path', 'type': 'string'},
  'start': {'default': 0, 'title': 'Start', 'type': 'integer'},
  'end': {'default': 0, 'title': 'End', 'type': 'integer'}},
 True)

Tool annotations distinguish read-only, destructive, and open-world calls. MCP clients use these fields to choose approval policy.

In [ ]:
mcp_w = server(host, readonly=False)
{t.name: (t.annotations.readOnlyHint, t.annotations.destructiveHint, t.annotations.openWorldHint)
 for t in await mcp_w.list_tools() if t.name in ('view_file', 'edit_file', 'web_search')}

{'view_file': (True, False, False),
 'edit_file': (False, True, False),
 'web_search': (True, False, True)}

Calling one goes through the same code path a turn does. What the client gets back is byte-for-byte what the model would have got:

In [ ]:
out = await mcp.call_tool('view_file', {'path': 'pkg/sizes.py'})
print(out[0][0].text if isinstance(out, tuple) else out)

1|d884|RESERVE = 16_384
2|0000|
3|b1c7|def threshold(ctx):
4|17f7|    return ctx - RESERVE


A disallowed write tool is absent from the MCP surface. Calling its name returns an error.

In [ ]:
with expect_fail(contains='Unknown tool'): await mcp.call_tool('edit_file', {'path': 'pkg/sizes.py', 'commands': '[]'})
[t.name for t in await mcp_w.list_tools() if t.name in WRITE_TOOLS]

['edit_file',
 'create_file',
 'edit_cell',
 'add_cell',
 'memory_forget',
 'run_python',
 'create_skill']

## Skills as resources

Skills are resources rather than tools, because that is what they are: text a client can read, not a call with an effect. The index is one resource and each skill is another. A client can list what is available and fetch only the one it needs. The same economy `read_skill` gives a model in a turn.

In [ ]:
[str(r.uriTemplate) for r in await mcp.list_resource_templates()] + [str(r.uri) for r in await mcp.list_resources()]

['skill://{name}', 'ramabana://skills', 'ramabana://status']

In [ ]:
print((await mcp.read_resource('ramabana://status'))[0].content[:200])

{
  "roots": [
    "/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpqh4_o51t/proj"
  ],
  "tools": [
    "search_code",
    "similar_code",
    "outline",
    "list_files",
    "view_file"


## The agent as one tool

With an agent passed in, the server also offers `ask`: a whole Ramabana turn behind a single call. The client spends one question and one answer. The tool loop, the tool results and the compaction all happen on this side and are discarded.

In [ ]:
agent, be = fake_agent(replies=['`threshold` is defined in pkg/sizes.py, and subtracts the reserve.'])
full = server(agent.host, agent)
[t.name for t in await full.list_tools() if t.name == 'ask']

['ask']

In [ ]:
out = await full.call_tool('ask', {'task': 'where is the threshold defined?'})
out[0][0].text if isinstance(out, tuple) else out

'`threshold` is defined in pkg/sizes.py, and subtracts the reserve.'

It runs on a worker thread, because a turn is blocking and an MCP server is asynchronous. The same split the terminal makes for the same reason.

In [ ]:
test_eq(len(be.sent), 1)
(await full.read_resource('ramabana://status'))[0].content[:60]

'{\n  "roots": [\n    "/proj"\n  ],\n  "tools": [\n    "search_cod'

## Running it

`ramabana-mcp` on the command line. `--model` is what turns the `ask` tool on: without a model there is nothing to delegate to. The server offers tools only.

In [ ]:
#| export
@call_parse
def main(
    root: str = '.',                 # folders to serve, comma separated
    model: str = None,               # the model `ask` runs on. Omit to serve tools only
    write: bool = False,             # mount the write tools too
    web: bool = True,                # let the web tools reach the network through fossick
    read_outside: bool = False,      # let reads name any path on this machine. Writes stay inside
    vault: bool = False,             # keep what is read in a vishalakshi vault, for the next session
    pii: str = PII_OFF,              # off | redact | refuse for what vault retrieval hands the model
    pii_ner: bool = False,           # --pii also gates titled names, not only patterns
    transport: str = 'stdio',        # stdio | sse | streamable-http
    cfg: str = None,                 # config dir, for skills and extensions
):
    "Serve Ramabana tools over MCP."
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    if model:
        agent, host = mk_agent(roots, model=model, approve='none', web=web, vault=vault,
                               read_outside=read_outside, pii=pii, pii_ner=pii_ner,
                               cfg=Path(cfg).expanduser() if cfg else None)
    else:
        agent, host = None, mk_host(roots, web=web, vault=vault, read_outside=read_outside,
                                    pii=pii, pii_ner=pii_ner)
    server(host, agent, readonly=not write).run(transport=transport)

In a client's configuration, over this repository, that is:

```json
{
  "mcpServers": {
    "ramabana": {
      "command": "ramabana-mcp",
      "args": ["--root", "/path/to/repo", "--model", "qwen-4b"]
    }
  }
}
```

Read-only, no model, is the default and the one to start with:

```bash
ramabana-mcp --root .
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()